# Colab runner — experiment 02
Runs the versioned Python implementation from the research branch. Existing runtime data is reused if present. On a fresh runtime the nano quickstart is regenerated. No tokens or raw data are committed.

In [ ]:
from pathlib import Path
import subprocess, os
repo = Path('/content/pragmatiq-scalable-explainable')
if not repo.exists():
    subprocess.run(['git','clone','-b','research','https://github.com/haider4445/pragmatiq-scalable-explainable.git',str(repo)],check=True)
os.chdir(repo)
subprocess.run(['git','switch','research'],check=True)
subprocess.run(['git','pull','--ff-only','origin','research'],check=True)

In [ ]:
%pip install -e '.[train]'

In [ ]:
import os
os.environ['PRAGMATIQ_DISABLE_FLASH']='1'
from pragmatiq import api
from pathlib import Path
root = Path('runs/research_baseline')
if not all((root / 'raw' / p).exists() for p in ['events.parquet','profiles.parquet']):
    print(api.quickstart(out=str(root), n_users=1000, model_size='nano', max_steps=30, seed=42, n_workers=0))

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'research/experiments/02_dataset_exploration.py'],check=True)

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display
print(Path('research/results/02_dataset_stats.json').read_text())
display(Image(filename='research/results/02_sequence_distribution.png'))

## Push results
Use your GitHub fine-grained token stored in Colab Secrets as GITHUB_TOKEN. Set Git identity once (use your GitHub noreply address if preferred). Review staged files before pushing. Do not print the token.

In [ ]:
from google.colab import userdata
import os, subprocess, shutil
assert shutil.which('gh'), 'Install GitHub CLI first if needed: !apt-get install -y gh'
env = dict(os.environ, GH_TOKEN=userdata.get('GITHUB_TOKEN'))
subprocess.run(['gh','auth','setup-git'],env=env,check=True)
subprocess.run(['git','config','user.name','haider4445'],check=True)
email = subprocess.check_output(['git','config','user.email'],text=True).strip() if subprocess.run(['git','config','user.email'],capture_output=True).returncode == 0 else ''
if not email:
    subprocess.run(['git','config','user.email','34925624+haider4445@users.noreply.github.com'],check=True)
files=['research/results/02_dataset_stats.json','research/results/02_sequence_distribution.png']
subprocess.run(['git','add','--',*files],check=True)
subprocess.run(['git','diff','--cached','--stat'],check=True)

In [ ]:
# Run after reviewing staged files; commit only this experiment's output.
subprocess.run(['git','commit','-m','experiment: record event length and padding waste results','--',*files],check=True)
subprocess.run(['git','push','origin','research'],env=env,check=True)
del env
subprocess.run(['git','log','-1','--oneline'],check=True)